# 🕵️ Dia 34 — Direito e OSINT, parte 2: armazenar, reter e descartar o que você coleta

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 34: OSINT law, part 2: storing, retaining, and disposing of what you collect* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-34.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Sexta, 16 de outubro. Os perfis falsos caíram, a hospedagem do site falso foi suspensa, os SMS pararam. O Dr. Hélio liga com uma pergunta só: *"A Vigia está guardando o quê, sobre quem, com que base, e até quando?"*

Rafa: *"Pacote de caso não é a pasta com tudo. É a pasta com o que precisa ficar, cifrada, com uma data de morte."*

Neste notebook você vai:

1. **Montar um caso de bancada** e reproduzir o `shasum -a 256 -c` em Python.
2. **Preencher o registro de retenção** da Isca e deixar o código checar cada linha.
3. **Minimizar um CSV** e provar por amostragem que a minimização foi aplicada.
4. **Ver o apagamento criptográfico** numa demonstração didática.
5. **Gerar o índice da Fase 2, o registro de descarte e o e-mail para o Dr. Hélio**.

🐍 Só biblioteca padrão. Tudo fictício. Material de estudo, não orientação jurídica.

## 1. Um caso de bancada, com hashes

Criamos uma pasta com arquivos **fictícios** que imitam o que a Fase 2 deixou, e o `hashes.txt` no mesmo formato do `shasum -a 256`.

In [ ]:
import hashlib
from pathlib import Path

CASO = Path("ISCA-P2")
ARQUIVOS = {
    "capturas/06-falso-inicio.html": "<html><title>Regularize sua chave Pix</title></html>",
    "capturas/03-serraazul-canais-oficiais.html": "<ul><li>Instagram: @serraazul</li></ul>",
    "prints/print-whatsapp-cooperado.csv": ("data,remetente_nome,remetente_tel,texto\n"
        "2026-10-05,Cooperado Fictício A,+55 00 00000-0091,Recebi este SMS e cliquei\n"
        "2026-10-05,Cooperada Fictícia B,+55 00 00000-0092,Vi a faixa da agência no grupo\n"),
    "img/points.csv": "name,lat,lon,method,confidence\naviso-agencia-centro,-20.15,-44.9,EXIF,contestado\n",
    "docs/meta.csv": "FileName,Producer,CreateDate\nComunicado_Oficial_SerraAzul.pdf,PDFRapido Online 4.0,2025:12:01\n",
    "infra/arestas-isca.csv": "Source,Target,Label\nserraazul-pix.example,serraazul-seguranca.example,SAN\n",
    "spiderfoot/export-emails.csv": "email\nti@serraazul.example\nfuncionario.ficticio@serraazul.example\n",
}
for nome, conteudo in ARQUIVOS.items():
    p = CASO / nome
    p.parent.mkdir(parents=True, exist_ok=True)       # cria as subpastas
    p.write_text(conteudo, encoding="utf-8")


def sha256_arquivo(caminho: Path) -> str:
    """Lê em blocos de 1 MiB: funciona igual para um CSV ou um disco de 50 GB."""
    h = hashlib.sha256()
    with open(caminho, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()


# = shasum -a 256 <arquivos> > hashes.txt   (formato: "<hash>  <caminho>")
linhas = [f"{sha256_arquivo(p)}  {p.relative_to(CASO)}" for p in sorted(CASO.rglob("*")) if p.is_file()]
(CASO / "hashes.txt").write_text("\n".join(linhas) + "\n", encoding="utf-8")
print((CASO / "hashes.txt").read_text())

In [ ]:
def verificar(pasta: Path, arquivo_hashes="hashes.txt"):
    """= shasum -a 256 -c hashes.txt. Devolve True se tudo OK."""
    tudo_ok = True
    for linha in (pasta / arquivo_hashes).read_text(encoding="utf-8").splitlines():
        if not linha.strip():
            continue
        esperado, nome = linha.split(maxsplit=1)
        alvo = pasta / nome.lstrip("*")               # shasum marca modo binário com *
        if not alvo.exists():
            print(f"{nome}: AUSENTE")                  # foi removido: precisa estar no registro de descarte
            tudo_ok = False
            continue
        ok = sha256_arquivo(alvo) == esperado
        tudo_ok &= ok
        print(f"{nome}: {'OK' if ok else 'FALHOU'}")
    return tudo_ok

print("Tudo OK?", verificar(CASO))

## 2. O registro de retenção da Operação Isca

As colunas do original, **mais** `base_lgpd` e `custodia` (vestígio que pode virar prova: CPP, art. 158-A e seguintes). Os valores são uma proposta; o exercício é você concordar ou mudar com motivo.

In [ ]:
import csv, io

REGISTRO = """conjunto|pessoal|de_quem|finalidade|base_lgpd|minimizacao|guardar_ate|descarte|custodia
capturas do site falso e do site real (Dia 19)|não|-|documentar o golpe|art. 7º IX (legítimo interesse)|nenhuma necessária|liberação pela autoridade|apagar com o contêiner|sim
prints de WhatsApp dos cooperados|sim|cooperados (nome, telefone)|comprovar o padrão da isca|art. 7º IX|nome e telefone mascarados na cópia de trabalho|fim do contrato + 90 dias|apagar com o contêiner|original fica com a cooperativa
aviso-agencia-centro.jpg e points.csv (Dia 27)|sim (nº de série, coordenada)|aparelho de quem fotografou|padrão: foto real + faixa falsa|art. 7º IX|nº de série só no pacote para a autoridade|liberação pela autoridade|apagar com o contêiner|sim
comunicado e rev1.pdf (Dia 28)|incidental (Author institucional)|cooperativa|provar reaproveitamento do documento|art. 7º IX|meta.csv sem nomes|liberação pela autoridade|apagar com o contêiner|sim
RDAP, crt.sh e DNS (Dias 29-30)|não|-|infraestrutura do golpe|n/a|nenhuma necessária|fim do contrato + 90 dias|apagar com o contêiner|sim
decisão e persona-P1 (Dia 31)|não|-|registro de decisão|n/a|nenhuma necessária|fim do contrato + 90 dias|apagar com o contêiner|não
norma de OPSEC e log ilustrativo (Dia 32)|não|-|procedimento interno|n/a|nenhuma necessária|sem data: documento interno vivo|n/a|não
export do SpiderFoot com e-mails (Dia 20)|sim|funcionários da cooperativa|nenhum achado depende dele|art. 7º IX|-|agora|apagar hoje e registrar|não
notas sobre exposições da intranet-antiga (Dia 23)|não (só URL e data)|-|comunicar a exposição à cooperativa|art. 7º IX|conteúdo nunca aberto|fim do contrato + 90 dias|apagar com o contêiner|não
"""

registro = list(csv.DictReader(io.StringIO(REGISTRO), delimiter="|"))

# Checagem do Checkpoint: toda linha tem data OU motivo para não ter; toda linha pessoal tem base e minimização.
problemas = 0
for r in registro:
    faltas = []
    if not r["guardar_ate"].strip():
        faltas.append("sem data de guarda nem motivo")
    if r["pessoal"].startswith("sim") and r["base_lgpd"] in ("", "n/a"):
        faltas.append("dado pessoal sem base legal")
    if r["pessoal"].startswith("sim") and r["minimizacao"] in ("", "-") and r["guardar_ate"] != "agora":
        faltas.append("dado pessoal sem minimização")
    problemas += bool(faltas)
    marca = "⚠️ " + "; ".join(faltas) if faltas else "✅"
    print(f"{marca:4s} {r['conjunto'][:55]:55s} até: {r['guardar_ate']}")
print(f"\n{len(registro)} linhas, {problemas} com problema.")

## 3. Minimizar agora, e provar que minimizou

O print do WhatsApp tem nome e telefone de cooperados. Para provar o **padrão** da isca, basta a data e o texto. Mascaramos na cópia de trabalho e depois fazemos a checagem por amostragem do Passo 7.

In [ ]:
import re, random

origem = CASO / "prints/print-whatsapp-cooperado.csv"
linhas = list(csv.DictReader(origem.open(encoding="utf-8")))


def mascarar_tel(t: str) -> str:
    """Mantém só os dois últimos dígitos: suficiente para conferir com o original, inútil para contatar."""
    return re.sub(r"\d(?=.*\d{2}$)", "•", t)


for l in linhas:
    l["remetente_nome"] = "[cooperado]"                 # o nome não sustenta nenhum achado
    l["remetente_tel"] = mascarar_tel(l["remetente_tel"])

with origem.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=linhas[0].keys())
    w.writeheader()
    w.writerows(linhas)
print(origem.read_text(encoding="utf-8"))

# Apaga o export do SpiderFoot (registro diz "agora")
alvo = CASO / "spiderfoot/export-emails.csv"
alvo.unlink()
print("🗑️ apagado:", alvo)

# ---- Checagem por amostragem (Passo 7): escolha uma linha COM dado pessoal ao acaso e confira ----
amostra = random.choice([r for r in registro if r["pessoal"].startswith("sim") and r["guardar_ate"] != "agora"])
print("\nAmostra:", amostra["conjunto"])
if "WhatsApp" in amostra["conjunto"]:
    texto = origem.read_text(encoding="utf-8")
    vazou = re.search(r"\+55 \d{2} \d{5}-\d{4}", texto) or "Fictício" in texto
    print("Minimização aplicada?", "NÃO ⚠️" if vazou else "sim ✅")
else:
    print("Confira à mão nos arquivos desta linha:", amostra["minimizacao"])

> Rode a verificação de hashes de novo: o print agora **FALHA** (você o alterou) e o export do SpiderFoot está **AUSENTE**. Isso não é erro, é o que tem que acontecer. Por isso a minimização se faz numa **cópia de trabalho** e cada alteração vai para o registro de descarte, com novo hash.

In [ ]:
verificar(CASO)
print("\n➡️ Regere o hashes.txt depois da minimização e guarde o antigo: os dois juntos contam a história.")
(CASO / "hashes-antes-da-minimizacao.txt").write_text((CASO / "hashes.txt").read_text(), encoding="utf-8")
novas = [f"{sha256_arquivo(p)}  {p.relative_to(CASO)}" for p in sorted(CASO.rglob("*"))
         if p.is_file() and not p.name.startswith("hashes")]
(CASO / "hashes.txt").write_text("\n".join(novas) + "\n", encoding="utf-8")
print("Depois de regerar:", verificar(CASO))

## 4. Apagamento criptográfico, explicado com um brinquedo

⚠️ **A cifra abaixo é didática e NÃO é segura.** Ela só existe para mostrar a ideia; para dados reais, use VeraCrypt, LUKS, FileVault ou BitLocker, como o original manda.

A ideia: se tudo o que está no disco está cifrado, apagar **a chave** torna todos os blocos inúteis, mesmo os que o SSD não apagou de verdade.

In [ ]:
import os, secrets


def fluxo(chave: bytes, n: int) -> bytes:
    """Gera n bytes pseudoaleatórios a partir da chave (SHA-256 em modo contador). BRINQUEDO."""
    out = bytearray()
    contador = 0
    while len(out) < n:
        out += hashlib.sha256(chave + contador.to_bytes(8, "big")).digest()
        contador += 1
    return bytes(out[:n])


def xor(dados: bytes, chave: bytes) -> bytes:
    return bytes(a ^ b for a, b in zip(dados, fluxo(chave, len(dados))))


chave = secrets.token_bytes(32)                       # a "frase-senha" do contêiner
claro = (CASO / "img/points.csv").read_bytes()
cifrado = xor(claro, chave)
Path("bloco-no-ssd.bin").write_bytes(cifrado)         # o que fica "esquecido" no SSD

print("No disco (cifrado):", cifrado[:40])
print("Com a chave       :", xor(Path("bloco-no-ssd.bin").read_bytes(), chave)[:40])

del chave                                             # = apagar o contêiner E a frase-senha
chute = secrets.token_bytes(32)
print("Sem a chave       :", xor(Path("bloco-no-ssd.bin").read_bytes(), chute)[:40])
print("\nO bloco continua no 'SSD', mas sem a chave é ruído. Registre a DATA em que a chave foi destruída.")

In [ ]:
# OPCIONAL (Linux): o mesmo princípio com ferramenta de verdade, num arquivo de teste (o teste pula linhas com !)
!apt-get -qq install -y cryptsetup > /dev/null
!cryptsetup --version
print("No Colab não dá para montar volumes cifrados; pratique o Passo 1 com o VeraCrypt na sua máquina.")

## 5. 📝 Índice da Fase 2, registro de descarte e e-mail ao Dr. Hélio

In [ ]:
#@title Fechamento da Fase 2 { display-mode: "form" }
pergunta_do_caso = "Qual é a infraestrutura do golpe do 'Pix bloqueado' que imita a Serra Azul, e qual o padrão das mensagens?"  #@param {type:"string"}
responsavel = "Analista júnior, Vigia"  #@param {type:"string"}
data_hoje = "2026-10-16"  #@param {type:"string"}

ARTEFATOS = {
    19: "plano + 10 capturas com hash", 20: "export filtrado SpiderFoot (bruto apagado)",
    21: "grafo Maltego", 22: "registro de reconhecimento", 23: "mapa de calor de documentos; exposições comunicadas, não abertas",
    24: "pegada própria (só minha)", 25: "linha do tempo social", 26: "linha do tempo de versões (Wayback)",
    27: "points.csv + notas do 'aviso'", 28: "rev1.pdf + anomalias", 29: "Gantt de registro",
    30: "arestas-isca.csv", 31: "decisão de persona (PARE)", 32: "norma de OPSEC", 33: "árvore de decisão + relatos",
    34: "registro de retenção + este índice",
}

indice = [f"# Fase 2 — Operação Isca (VIGIA-2026-001)", "", f"**Pergunta do caso (Dia 19):** {pergunta_do_caso}", "",
          "## Artefatos por dia"]
indice += [f"- Dia {d}: {a}" for d, a in ARTEFATOS.items()]
indice += ["", "## Registro de retenção", "Ver `registro-de-retencao.csv` neste contêiner.", "",
           "## Registro de descarte", "| Conjunto | Data | Método | Por quem |", "|---|---|---|---|",
           f"| export do SpiderFoot com e-mails (Dia 20) | {data_hoje} | apagado (dentro do contêiner cifrado) | {responsavel} |",
           f"| nomes e telefones nos prints de WhatsApp (cópia de trabalho) | {data_hoje} | mascarados; hash novo registrado | {responsavel} |",
           "| (contêiner inteiro) | (data de liberação) | apagamento criptográfico: contêiner + frase-senha | |"]
(CASO / "README.md").write_text("\n".join(indice) + "\n", encoding="utf-8")

with (CASO / "registro-de-retencao.csv").open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=registro[0].keys())
    w.writeheader()
    w.writerows(registro)

email = f"""Dr. Hélio,

A Vigia guarda, para o caso VIGIA-2026-001, {len(registro)} conjuntos de dados listados no registro de retenção anexo. Os únicos com dados pessoais são os prints enviados pelos cooperados (nomes e telefones mascarados na nossa cópia de trabalho) e o número de série do aparelho na foto do "aviso", reservado ao pacote para a autoridade. A base é o legítimo interesse da cooperativa na prevenção à fraude (LGPD, art. 7º, IX), registrada conforme o art. 37. O que pode servir de prova fica sob cadeia de custódia até liberação pela autoridade; o restante, até 90 dias após o fim do contrato. Tudo está num contêiner cifrado, e o descarte será por apagamento criptográfico, com data registrada. O export com e-mails de funcionários foi eliminado hoje ({data_hoje}).

Atenciosamente,
{responsavel}
"""
print(email)
(CASO / "email-dr-helio.txt").write_text(email, encoding="utf-8")
try:
    from google.colab import files
    import shutil
    shutil.make_archive("ISCA-P2", "zip", CASO)
    files.download("ISCA-P2.zip")
except ImportError:
    print("(Fora do Colab: tudo ficou na pasta ISCA-P2/.)")

## 6. 🎮 Quiz: fica, minimiza ou sai?

**F** = fica como está · **M** = minimiza · **S** = sai hoje (com linha no registro de descarte). `MODO_JOGO = False` mostra o gabarito.

In [ ]:
QUIZ = [
    ("hashes.txt dos arquivos do caso", "F", "Sem dado pessoal, e é o que dá confiança ao resto."),
    ("Banco bruto da varredura do SpiderFoot, depois de exportar o filtro usado", "S", "Guarda tudo, inclusive o ruído."),
    ("Print de WhatsApp com nome e telefone do cooperado", "M", "O padrão precisa do texto e da data, não do nome."),
    ("Comunicado PDF com três revisões", "F", "Vestígio de prova: fica sob custódia até liberação."),
    ("Ponto do mapa do Dia 27 a 300 m da sua casa", "S", "O original manda remover pontos perto de onde você mora."),
    ("meta.csv com coluna Author de pessoas", "M", "Descarte as colunas de nomes; o padrão não precisa delas."),
    ("Uma cópia não cifrada do caso no Desktop 'para facilitar'", "S", "Mova para o contêiner, verifique e apague; registre."),
    ("Notas com a URL das exposições da intranet-antiga (conteúdo nunca aberto)", "F", "Só URL e data; é o que a cooperativa precisa para corrigir."),
]

MODO_JOGO = True
acertos = 0
for i, (item, certo, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}/{len(QUIZ)}] {item}")
    if MODO_JOGO:
        r = input("   F / M / S: ").strip().upper()[:1]
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end=" ")
    print(f"Gabarito: {certo}. {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## ✅ Antes de fechar o Dia 34 (e a Fase 2)

- [ ] Registro de retenção com `base_lgpd` e `custodia` em todas as linhas
- [ ] Minimização aplicada e conferida por amostragem
- [ ] Hashes regerados depois da minimização, com o antigo guardado
- [ ] Contêiner VeraCrypt: desmontado = ilegível; montado = abre
- [ ] `README.md` índice + registro de descarte
- [ ] E-mail para o Dr. Hélio

**Fase 3 (Dia 35):** o IscaKit reaparece contra outras quatro cooperativas. A pergunta muda de "o que é este site?" para "quem é este padrão?", e a resposta vai ser um cluster de atividade, nunca uma pessoa.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>